In [2]:
# Cell 1 — Imports
import asyncpg
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

In [3]:
# Cell 2 — Connection
DB_CONFIG = {
    "host": "host-db",
    "port": 5432,
    "database": "db",
    "user": "user",
    "password": "pass",
}

pool = await asyncpg.create_pool(**DB_CONFIG, min_size=2, max_size=10)
print("connected")

connected


In [4]:
# Cell 3 — User profile from interactions
USER_ID = 1

rows = await pool.fetch("""
    SELECT
        group_id,
        COUNT(*) FILTER (WHERE interaction_type = 'upvote')   AS upvotes,
        COUNT(*) FILTER (WHERE interaction_type = 'comment')  AS comments,
        COUNT(*) FILTER (WHERE interaction_type = 'view')     AS views,
        COUNT(*) FILTER (WHERE interaction_type = 'title')    AS title_clicks,
        COUNT(*)                                               AS total,
        EXTRACT(EPOCH FROM (NOW() - MAX(timestamp))) / 86400.0 AS days_since_last
    FROM interactions
    WHERE user_id = $1
    GROUP BY group_id
""", USER_ID)

profile_df = pd.DataFrame([dict(r) for r in rows])
profile_df = profile_df.astype({
    "group_id":       int,
    "upvotes":        float,
    "comments":       float,
    "views":          float,
    "title_clicks":   float,
    "total":          float,
    "days_since_last": float,
})
profile_df["recency_decay"]    = np.exp(-profile_df["days_since_last"] / 30)
profile_df["engagement_ratio"] = (
    (profile_df["upvotes"] + profile_df["comments"] + profile_df["title_clicks"])
    / profile_df["total"].replace(0, np.nan)
).fillna(0)

print(f"Groups in profile: {len(profile_df)}")
profile_df.head()

Groups in profile: 58


,group_id,upvotes,comments,views,title_clicks,total,days_since_last,recency_decay,engagement_ratio
0,113899,0.0,20.0,206.0,13.0,240.0,0.158805,0.994720,0.137500
1,32231,0.0,27.0,295.0,8.0,330.0,0.158647,0.994726,0.106061
2,6250,0.0,44.0,703.0,35.0,783.0,0.151903,0.994949,0.100894
3,88848,0.0,13.0,1135.0,45.0,1196.0,0.150702,0.994989,0.048495
4,61865,0.0,16.0,434.0,49.0,500.0,0.158647,0.994726,0.130000


In [5]:
# Cell 4 — Candidate submissions (unseen, non-nsfw, last 72h)
rows = await pool.fetch("""
    SELECT
        s.id,
        s.title,
        s.group_id,
        g.name                                              AS group_name,
        s.score::float                                      AS score,
        s.num_comments::float                               AS num_comments,
        EXTRACT(EPOCH FROM (NOW() - s.created)) / 3600.0   AS hours_ago
    FROM submissions s
    JOIN groups g ON g.id = s.group_id
    WHERE NOT EXISTS (
        SELECT 1 FROM interactions i
        WHERE i.submission_id = s.id
          AND i.user_id = $1
    )
    AND s.created >= NOW() - INTERVAL '72 hours'
    AND s.is_nsfw  = FALSE
    AND g.is_nsfw  = FALSE
    ORDER BY s.score DESC, s.created DESC
    LIMIT 1000
""", USER_ID)

candidates_df = pd.DataFrame([dict(r) for r in rows])
candidates_df = candidates_df.astype({
    "group_id":    int,
    "score":       float,
    "num_comments": float,
    "hours_ago":   float,
})
print(f"Candidates: {len(candidates_df)}")
candidates_df.head()

Candidates: 1000


,id,title,group_id,group_name,score,num_comments,hours_ago
0,1sknt54,"you shouldn't have been bitin' my horsey, boy.",61865,interestingasfuck,64816.0,2188.0,8.641549
1,1sjo5om,Hungary election: Orbán concedes to Magyar's T...,84702,news,39597.0,1385.0,34.026549
2,1skd6md,"Trump unable to name one verse from ""favorite ...",131316,videos,23435.0,1735.0,14.703771
3,1sk10ie,"Trump, 79, Posts Himself as Christ After Bonke...",96078,politics,22256.0,1909.0,24.723771
4,1skp3fq,New poll pegs Mark Kelly as a leading 2028 pre...,96078,politics,20247.0,3467.0,7.860716


In [6]:
# Cell 5 — Training labels (seen submissions, did user engage?)
rows = await pool.fetch("""
    SELECT
        s.id                                                        AS submission_id,
        s.group_id,
        s.score::float                                              AS score,
        s.num_comments::float                                       AS num_comments,
        EXTRACT(EPOCH FROM (NOW() - s.created)) / 3600.0           AS hours_ago,
        MAX(CASE
            WHEN i.interaction_type IN ('upvote', 'comment', 'title')
            THEN 1 ELSE 0
        END)                                                        AS label
    FROM interactions i
    JOIN submissions s ON s.id = i.submission_id
    WHERE i.user_id = $1
      AND s.is_nsfw = FALSE
    GROUP BY s.id, s.group_id, s.score, s.num_comments, s.created
""", USER_ID)

labels_df = pd.DataFrame([dict(r) for r in rows])
labels_df = labels_df.astype({
    "group_id":    int,
    "score":       float,
    "num_comments": float,
    "hours_ago":   float,
    "label":       int,
})
print(f"Labeled submissions : {len(labels_df)}")
print(f"Positive (engaged)  : {labels_df['label'].sum()}")
print(f"Negative (view only): {(labels_df['label'] == 0).sum()}")
print(f"Engagement rate     : {labels_df['label'].mean():.2%}")
labels_df.head()

Labeled submissions : 24101
Positive (engaged)  : 3591
Negative (view only): 20510
Engagement rate     : 14.90%


,submission_id,group_id,score,num_comments,hours_ago,label
0,1r8avhd,114538,1566.0,130.0,1307.221274,0
1,1rkasxf,10840,341.0,31.0,986.003774,0
2,1sdkukb,113964,1025.0,92.0,196.354608,0
3,1rlvhrg,119957,8434.0,278.0,943.384885,0
4,1rafwxj,88848,756.0,84.0,1250.561552,0


In [7]:
# Cell 6 — Build training feature matrix
train = labels_df.copy()

train["normalized_score"]     = train["score"]        / train["score"].max()
train["normalized_comments"]  = train["num_comments"] / train["num_comments"].replace(0, np.nan).max()
train["submission_freshness"] = np.exp(-train["hours_ago"].astype(float) / 24)

train = train.merge(
    profile_df[["group_id", "upvotes", "comments", "engagement_ratio", "recency_decay"]],
    on="group_id",
    how="left",
)
train["is_known_group"] = train["upvotes"].notna().astype(int)
train.fillna(0, inplace=True)

FEATURE_COLS = [
    "normalized_score",
    "normalized_comments",
    "submission_freshness",
    "engagement_ratio",
    "recency_decay",
    "is_known_group",
]

# ensure all float
train[FEATURE_COLS] = train[FEATURE_COLS].astype(float)

print(f"Training matrix: {train.shape}")
train.head()

Training matrix: (24101, 14)


,submission_id,group_id,score,num_comments,hours_ago,label,normalized_score,normalized_comments,submission_freshness,upvotes,comments,engagement_ratio,recency_decay,is_known_group
0,1r8avhd,114538,1566.0,130.0,1307.221274,0,0.009469,0.007374,2.213310e-24,0.0,5.0,0.030137,0.991117,1.0
1,1rkasxf,10840,341.0,31.0,986.003774,0,0.002062,0.001758,1.437695e-18,0.0,1.0,0.008011,0.995002,1.0
2,1sdkukb,113964,1025.0,92.0,196.354608,0,0.006198,0.005218,2.797982e-04,0.0,15.0,0.145669,0.994976,1.0
3,1rlvhrg,119957,8434.0,278.0,943.384885,0,0.050997,0.015769,8.489483e-18,0.0,175.0,0.143775,0.995002,1.0
4,1rafwxj,88848,756.0,84.0,1250.561552,0,0.004571,0.004765,2.346033e-23,0.0,13.0,0.048495,0.994989,1.0


In [8]:
# Cell 7 — Train LightGBM
X = train[FEATURE_COLS]
y = train["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scale = (y_train == 0).sum() / (y_train == 1).sum()

model = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=6,
    scale_pos_weight=scale,
    random_state=42,
    verbose=-1,
)

model.fit(X_train, y_train, eval_set=[(X_test, y_test)])

y_proba = model.predict_proba(X_test)[:, 1]
print(f"ROC-AUC : {roc_auc_score(y_test, y_proba):.4f}")
print(classification_report(y_test, model.predict(X_test)))

ROC-AUC : 0.7659
              precision    recall  f1-score   support

           0       0.93      0.70      0.80      4103
           1       0.29      0.70      0.41       718

    accuracy                           0.70      4821
   macro avg       0.61      0.70      0.60      4821
weighted avg       0.83      0.70      0.74      4821



In [9]:
# Cell 8 — Feature importance
pd.DataFrame({
    "feature":    FEATURE_COLS,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=False)

,feature,importance
2,submission_freshness,837
1,normalized_comments,682
0,normalized_score,599
3,engagement_ratio,517
4,recency_decay,303
5,is_known_group,0


In [14]:
# Cell 9 — Rank candidate submissions
cands = candidates_df.copy()

cands["normalized_score"]     = cands["score"]        / cands["score"].max()
cands["normalized_comments"]  = cands["num_comments"] / cands["num_comments"].replace(0, np.nan).max()
cands["submission_freshness"] = np.exp(-cands["hours_ago"].astype(float) / 24)

cands = cands.merge(
    profile_df[["group_id", "upvotes", "comments", "engagement_ratio", "recency_decay"]],
    on="group_id",
    how="left",
)
cands["is_known_group"] = cands["upvotes"].notna().astype(int)
cands.fillna(0, inplace=True)
cands[FEATURE_COLS] = cands[FEATURE_COLS].astype(float)

cands["engagement_score"] = model.predict_proba(cands[FEATURE_COLS])[:, 1]
ranked = cands.sort_values("engagement_score", ascending=False).reset_index(drop=True)
ranked.index += 1

ranked[["id", "title", "group_name", "score", "hours_ago", "engagement_score"]].head(100)

,id,title,group_name,score,hours_ago,engagement_score
1,1sj8ejb,"Bishara Bahbah, Chair of “Arab Americans for T...",LeopardsAteMyFace,1727.0,45.582938,0.855991
2,1sjj5u6,Fighting 'price gouging' by... starting a war ...,LeopardsAteMyFace,2763.0,37.110438,0.851766
3,1sipm1z,Hezbollah drone intercepted over northern Isra...,CombatFootage,836.0,59.886827,0.844753
4,1sje5zt,High-res footage of a Sting interceptor drone ...,CombatFootage,1727.0,40.410438,0.842785
5,1sj80l0,‘Everything is gone’: Israel destroys entire v...,worldnews,583.0,45.952660,0.839381
6,1sivl58,FPV kamikaze drones hit multiple Russian soldi...,CombatFootage,612.0,56.053494,0.836179
7,1skepf1,IDF eliminates two hezbullah fighters | oct 30...,CombatFootage,356.0,13.841827,0.830799
8,1sio191,Trump says US 'completely destroyed' Iran's mi...,worldnews,626.0,60.869327,0.828514
9,1sjlr0z,IDF kills Hamas terrorists involved in hostage...,worldnews,1010.0,35.509883,0.825590
10,1sijsvb,“Trump knows what he’s doing. He knows exactly...,LeopardsAteMyFace,1256.0,63.633216,0.824894


In [16]:
# Cell 10 — Re-rank with diversity penalty
cands_reranked = ranked.copy()

# within each group, assign a rank (1st post from group, 2nd post, 3rd post...)
cands_reranked["group_rank"] = (
    cands_reranked.groupby("group_name")["engagement_score"]
    .rank(ascending=False, method="first")
)

# penalize each subsequent post from the same group exponentially
# 1st post: penalty = 1.0, 2nd: 0.5, 3rd: 0.25, 4th: 0.125 etc.
cands_reranked["diversity_penalty"] = 0.5 ** (cands_reranked["group_rank"] - 1)

# final score = model score * penalty
cands_reranked["final_score"] = (
    cands_reranked["engagement_score"] * cands_reranked["diversity_penalty"]
)

cands_reranked = (
    cands_reranked
    .sort_values("final_score", ascending=False)
    .reset_index(drop=True)
)
cands_reranked.index += 1

cands_reranked[["id", "title", "group_name", "score", "hours_ago", "engagement_score", "final_score"]].head(30)

,id,title,group_name,score,hours_ago,engagement_score,final_score
1,1sj8ejb,"Bishara Bahbah, Chair of “Arab Americans for T...",LeopardsAteMyFace,1727.0,45.582938,0.855991,0.855991
2,1sipm1z,Hezbollah drone intercepted over northern Isra...,CombatFootage,836.0,59.886827,0.844753,0.844753
3,1sj80l0,‘Everything is gone’: Israel destroys entire v...,worldnews,583.0,45.952660,0.839381,0.839381
4,1sj0u5d,Arkady Tsaregradtsev continues to drift while ...,Drifting,918.0,52.167105,0.803976,0.803976
5,1sjpdjr,Help? Not sure what this Manga is,Ghost_in_the_Shell,307.0,33.277105,0.797325,0.797325
6,1sjfgrl,Ferrari crashes into utility pole,PublicFreakout,1067.0,39.493494,0.795831,0.795831
7,1sjifnw,Google is now letting you change your Gmail ad...,technology,880.0,37.563771,0.791350,0.791350
8,1sj5h9a,Chinese opera is on next freaking level!,nextfuckinglevel,413.0,48.312660,0.780741,0.780741
9,1sjh5hp,Iran 'Flow Control': How Tehran Took the Globa...,Economics,329.0,38.379327,0.774832,0.774832
10,1sjd2b1,IRAN-US talks in Islamabad end,news,703.0,41.248494,0.770007,0.770007


In [18]:
# tweak these two values to adjust feed feel
DIVERSITY_PENALTY = 0.5   # lower = more diverse, higher = more of user's top groups
FRESHNESS_WEIGHT  = 2.0   # multiply hours_ago decay to punish old posts harder

cands_reranked["submission_freshness_boosted"] = np.exp(
    -cands_reranked["hours_ago"].astype(float) / 24 * FRESHNESS_WEIGHT
)
cands_reranked["final_score"] = (
    cands_reranked["engagement_score"]
    * cands_reranked["diversity_penalty"]
    * cands_reranked["submission_freshness_boosted"]
)

cands_reranked = (
    cands_reranked
    .sort_values("final_score", ascending=False)
    .reset_index(drop=True)
)
cands_reranked.index += 1
cands_reranked[["id", "title", "group_name", "score", "hours_ago", "engagement_score", "final_score"]].head(100)

,id,title,group_name,score,hours_ago,engagement_score,final_score
1,1skimoy,Avi Lewis says high-speed rail project should ...,CanadaPolitics,383.0,11.608494,0.760720,0.289134
2,1skkq5u,Punishment or fetish activities in the Russian...,UkraineWarVideoReport,471.0,10.407383,0.652251,0.274005
3,1skqe98,"Despite Apocalyptic Warnings, California Fast ...",Economics,1275.0,7.014327,0.743360,0.207163
4,1skaw4t,Magyar: Ukraine must go through the entire neg...,UkrainianConflict,484.0,16.089049,0.765635,0.200327
5,1ska0pu,"Apophis will pass closest to Earth on Friday, ...",spaceporn,354.0,16.657383,0.686746,0.171374
6,1skvw8z,Official Posters for 'Spider-Man: Brand New Day',movies,4417.0,3.084049,0.402609,0.155682
7,1skc81s,That ‘quantum heartbeat detector’ allegedly us...,TrueReddit,250.0,15.273771,0.544471,0.152475
8,1skvwcw,Kitty Mafia here! Everybody step back!,aww,3476.0,3.081827,0.196549,0.152032
9,1sk9axs,When Churchill had more sympathy for Nazi offi...,HistoryMemes,3681.0,17.157660,0.611990,0.146483
10,1skwwm4,The fish gave her a reason to cry about 😭,funny,832.0,2.319883,0.156459,0.128956
